In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import matplotlib.font_manager as fm
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler

In [2]:
#Colour palette
 
BG          = "#f5f4f0"
CARD        = "#ffffff"
CARD_BORDER = "#d9d8d2"
TEAL        = "#1D9E75"
CORAL       = "#D85A30"
BLUE        = "#5C9EE0"
GRAY        = "#888780"
TEXT_PRIMARY   = "#1a1a18"
TEXT_SECONDARY = "#5f5e5a"
TEXT_LABEL     = "#888780"
 
# Font stack — fall back gracefully if system fonts aren't installed
FONT_CANDIDATES = ["Helvetica Neue", "Arial", "DejaVu Sans"]
AVAILABLE_FONTS = {f.name for f in fm.fontManager.ttflist}
FONT_FAMILY = next((f for f in FONT_CANDIDATES if f in AVAILABLE_FONTS), "Arial")
 
plt.rcParams.update({
    "font.family":      FONT_FAMILY,
    "text.color":       TEXT_PRIMARY,
    "axes.edgecolor":   CARD_BORDER,
    "axes.labelcolor":  TEXT_SECONDARY,
    "xtick.color":      TEXT_SECONDARY,
    "ytick.color":      TEXT_SECONDARY,
    "figure.facecolor": BG,
    "savefig.facecolor": BG,
})
 

In [3]:
# Helper

def style_axis(ax, title=None, subtitle=None):
    """Apply consistent minimal-axis styling chart-box look."""
    ax.set_facecolor(CARD)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    for spine in ["left", "bottom"]:
        ax.spines[spine].set_color(CARD_BORDER)
    ax.tick_params(length=0, labelsize=8)
    ax.grid(axis="y", color="#FAFAFA", linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    if title:
        ax.set_title(title, fontsize=10.5, fontweight=600, color=TEXT_PRIMARY,
                     loc="left", pad=20)
    if subtitle:
        ax.text(0, 1.1, subtitle, transform=ax.transAxes, fontsize=7.5,
                color=TEXT_LABEL, va="bottom")

In [4]:
# Data loading
 
df = pd.read_csv("Data for Task 1.csv")
df = df.drop(columns=[c for c in ["id", "Unnamed: 32"] if c in df.columns])

def load_data(path):
    df = pd.read_csv(path)
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
    return df


In [5]:
# Panel 1: Donut chart — class balance
path = "diagnosis_split.png" 
counts = df["diagnosis"].value_counts()

benign = counts.get("B", 0)
malignant = counts.get("M", 0)
total = benign + malignant

labels = [
    f"Benign\n({benign/total:.1%})",
    f"Malignant\n({malignant/total:.1%})"
]

fig, ax = plt.subplots(figsize=(7,6))

wedges, texts = ax.pie(
    [benign, malignant],
    labels=labels,
    colors=[TEAL, CORAL],
    radius=0.8,
    startangle=90,
    labeldistance=1.07,
    textprops={
        "fontsize": 10
    },
    wedgeprops=dict(width=0.4, edgecolor="white", linewidth=3)
)

ax.set_title("Diagnosis Split", y=0.85)
fig.tight_layout()
fig.savefig(path, dpi=180, facecolor=BG, bbox_inches="tight")
plt.close(fig)
plt.show()

In [6]:
# Panel 2: Horizontal bar — % difference malignant vs benign
path = "diff_bar.png"

fig, ax = plt.subplots(figsize=(9,6))

feat_cols = [c for c in df.columns if c.endswith("_mean") and c != "diagnosis"]

m = df[df.diagnosis == "M"][feat_cols].mean()
b = df[df.diagnosis == "B"][feat_cols].mean()
pct_diff = ((m - b) / b * 100)

labels_map = {
    "concavity_mean": "Concavity",
    "concave points_mean": "Concave points",
    "area_mean": "Area",
    "compactness_mean": "Compactness",
    "perimeter_mean": "Perimeter",
    "radius_mean": "Radius",
    "texture_mean": "Texture",
    "smoothness_mean": "Smoothness",
    "symmetry_mean": "Symmetry",
    "fractal_dimension_mean": "Fractal dim.",
}

order = [
    "concavity_mean",
    "concave points_mean",
    "area_mean",
    "compactness_mean",
    "perimeter_mean",
    "radius_mean",
    "texture_mean",
    "smoothness_mean",
    "symmetry_mean",
    "fractal_dimension_mean"
]

order = [f for f in order if f in pct_diff.index]

vals = pct_diff[order].values
labels = [labels_map.get(f, f) for f in order]

colors = [CORAL if v >= 5 else GRAY for v in vals]
y_pos = np.arange(len(labels))[::-1]

ax.barh(y_pos, vals, color=colors, height=0.5, zorder=2)

ax.set_yticks(y_pos)
ax.set_yticklabels(labels, fontsize=16)

ax.set_xlabel("% difference", fontsize=8, color=TEXT_LABEL)
ax.xaxis.set_major_formatter(lambda v, pos: f"{int(v)}%")

style_axis(ax, "How much higher are malignant values?", None)

ax.text(
    0, 1.015,
    "% difference (malignant vs benign average)",
    transform=ax.transAxes,
    fontsize=7.5,
    color=TEXT_LABEL,
    va="bottom"
)

handles = [
    mpatches.Patch(color=CORAL, label="Malignant higher"),
    mpatches.Patch(color=GRAY, label="No meaningful difference")
]

ax.legend(
    handles=handles,
    loc="lower right",
    frameon=False,
    fontsize=7.5,
    labelcolor=TEXT_SECONDARY
)

fig.tight_layout()
fig.savefig(path, dpi=180, facecolor=BG, bbox_inches="tight")
plt.close(fig)

In [7]:
# Panel 3: Grouped histogram
 
def plot_feature_histogram(ax, df, feature, n_bins=11):
    m_vals = df[df.diagnosis == "M"][feature].dropna()
    b_vals = df[df.diagnosis == "B"][feature].dropna()
    all_vals = df[feature].dropna()
 
    bins = np.linspace(all_vals.min(), all_vals.max(), n_bins)
    bin_centers = (bins[:-1] + bins[1:]) / 2
    width = (bins[1] - bins[0]) * 0.42
 
    b_hist, _ = np.histogram(b_vals, bins=bins)
    m_hist, _ = np.histogram(m_vals, bins=bins)
 
    ax.bar(bin_centers - width/2, b_hist, width=width, color=TEAL, alpha=0.85,
          label="Benign", zorder=2)
    ax.bar(bin_centers + width/2, m_hist, width=width, color=CORAL, alpha=0.85,
          label="Malignant", zorder=2)
 
    b_mean, m_mean = b_vals.mean(), m_vals.mean()
 
    def fmt(v):
        if abs(v) >= 100: return f"{v:.0f}"
        if abs(v) >= 10:  return f"{v:.1f}"
        if abs(v) >= 1:   return f"{v:.2f}"
        return f"{v:.4f}"
 
    title = feature.replace("_", " ")
    
    # Main title
    ax.set_title(title,
                 loc="left",
                 fontsize=12,
                 fontweight="bold",
                 pad=28)
    
    # Subtitle directly below title
    subtitle = f"Benign avg: {fmt(b_mean)}    Malignant avg: {fmt(m_mean)}"
    
    ax.text(0, 1.02, subtitle,
            transform=ax.transAxes,
            ha="left",
            va="bottom",
            fontsize=9,
            color="gray")
    ax.tick_params(axis="x", labelsize=7)
    ax.tick_params(axis="y", labelsize=7)
 
 
def save_single_feature_histogram(df, feature, save_path, figsize=(6, 4)):
    fig, ax = plt.subplots(figsize=figsize, facecolor=BG)

    plot_feature_histogram(ax, df, feature)

    fig.tight_layout()
    fig.savefig(save_path, dpi=180, facecolor=BG, bbox_inches="tight")
    plt.close(fig)

    print(f"Saved: {save_path}")


def build_feature_histogram_images(df, save_prefix="histogram"):
    groups = {
        "mean":  [c for c in df.columns if c.endswith("_mean")],
        "se":    [c for c in df.columns if c.endswith("_se")],
        "worst": [c for c in df.columns if c.endswith("_worst")],
    }

    for group_name, features in groups.items():
        for feature in features:
            clean_name = feature.replace(" ", "_")
            path = f"{save_prefix}_{group_name}_{clean_name}.png"

            save_single_feature_histogram(
                df=df,
                feature=feature,
                save_path=path
            )

In [8]:
# Entry point

if __name__ == "__main__":
    df = load_data("Data for Task 1.csv")

    build_feature_histogram_images(df, save_prefix="histogram")


Saved: histogram_mean_radius_mean.png
Saved: histogram_mean_texture_mean.png
Saved: histogram_mean_perimeter_mean.png
Saved: histogram_mean_area_mean.png
Saved: histogram_mean_smoothness_mean.png
Saved: histogram_mean_compactness_mean.png
Saved: histogram_mean_concavity_mean.png
Saved: histogram_mean_concave_points_mean.png
Saved: histogram_mean_symmetry_mean.png
Saved: histogram_mean_fractal_dimension_mean.png
Saved: histogram_se_radius_se.png
Saved: histogram_se_texture_se.png
Saved: histogram_se_perimeter_se.png
Saved: histogram_se_area_se.png
Saved: histogram_se_smoothness_se.png
Saved: histogram_se_compactness_se.png
Saved: histogram_se_concavity_se.png
Saved: histogram_se_concave_points_se.png
Saved: histogram_se_symmetry_se.png
Saved: histogram_se_fractal_dimension_se.png
Saved: histogram_worst_radius_worst.png
Saved: histogram_worst_texture_worst.png
Saved: histogram_worst_perimeter_worst.png
Saved: histogram_worst_area_worst.png
Saved: histogram_worst_smoothness_worst.png
Save

In [9]:
# Feature separation boxplots

top_features = [
    "radius_worst",
    "concave points_worst",
    "compactness_worst",
    "concave points_mean",
    "radius_mean",
    "compactness_mean"
]

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    ax = axes[i]
    benign_vals    = df[df["diagnosis"] == "B"][feat] 
    malignant_vals = df[df["diagnosis"] == "M"][feat]

    bp = ax.boxplot(
        [benign_vals, malignant_vals],
        labels=["Benign", "Malignant"],
        patch_artist=True,
        medianprops=dict(color="white", linewidth=2.5),
        whiskerprops=dict(color="#666666"),
        capprops=dict(color="#666666"),
        flierprops=dict(marker="o", markerfacecolor="#999999",
                        markersize=3, alpha=0.5, linestyle="none")
    )
    bp["boxes"][0].set_facecolor(TEAL)
    bp["boxes"][1].set_facecolor(CORAL)

    ax.set_title(feat.replace("_", " ").title(), fontsize=10, fontweight="bold")
    ax.set_ylabel("Value", fontsize=9)
    ax.yaxis.grid(True, linestyle="--", alpha=0.4)
    ax.set_axisbelow(True)

fig.suptitle("Key Feature Distributions by Diagnosis",
             fontsize=10, fontweight="bold", y=1.01)

legend_patches = [
    mpatches.Patch(color=TEAL, label="Benign"),
    mpatches.Patch(color=CORAL,   label="Malignant")
]
fig.legend(handles=legend_patches, loc="upper right",
           bbox_to_anchor=(1.0, 1.0), fontsize=9)
plt.tight_layout()
plt.savefig("fig1_feature_boxplots.png", dpi=150, bbox_inches="tight")
plt.close()
print("Fig 1 saved")

# Correlation heatmap (mean features)
mean_features = [c for c in df.columns if c.endswith("_mean") and c != "diagnosis"]
corr = df[mean_features].corr()

mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(10, 8))
cmap = sns.diverging_palette(220, 20, as_cmap=True)
sns.heatmap(corr, mask=mask, cmap=cmap, center=0,
            vmin=-1, vmax=1,
            annot=True, fmt=".2f", annot_kws={"size": 8},
            linewidths=0.5, linecolor="white",
            square=True, ax=ax,
            cbar_kws={"shrink": 0.8, "label": "Pearson r"})
ax.set_title("Correlation Matrix — Mean Features\n(Coral = high multicollinearity → features dropped)",
             fontsize=10, fontweight="bold", pad=12)
labels = [l.get_text().replace("_mean", "") for l in ax.get_xticklabels()]
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(labels, rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig("fig2_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.close()
print("Fig 2 saved")


C:\Users\New\AppData\Local\Temp\ipykernel_11348\2910189557.py:20: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(
C:\Users\New\AppData\Local\Temp\ipykernel_11348\2910189557.py:20: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(
C:\Users\New\AppData\Local\Temp\ipykernel_11348\2910189557.py:20: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(
C:\Users\New\AppData\Local\Temp\ipykernel_11348\2910189557.py:20: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  b

Fig 1 saved
Fig 2 saved


In [10]:
y = (df["diagnosis"] == "M").astype(int)

corrs = (
    df.drop(columns=["diagnosis"])
      .corrwith(y)
      .abs()
      .sort_values(ascending=False)
)

print(corrs.head(10))
print(corrs.tail(10))


concave points_worst    0.793566
perimeter_worst         0.782914
concave points_mean     0.776614
radius_worst            0.776454
perimeter_mean          0.742636
area_worst              0.733825
radius_mean             0.730029
area_mean               0.708984
concavity_mean          0.696360
concavity_worst         0.659610
dtype: float64
symmetry_mean              0.330499
fractal_dimension_worst    0.323872
compactness_se             0.292999
concavity_se               0.253730
fractal_dimension_se       0.077972
smoothness_se              0.067016
id                         0.039769
fractal_dimension_mean     0.012838
texture_se                 0.008303
symmetry_se                0.006522
dtype: float64


In [11]:
# Remove target
df = df.drop(columns=[c for c in ["id", "Unnamed: 32"] if c in df.columns])
X = df.drop(columns=["diagnosis"])


# Calculate VIF
vif_df = pd.DataFrame({
    "Feature": X.columns,
    "VIF": [variance_inflation_factor(X.values, i)
            for i in range(X.shape[1])]
})

vif_df = vif_df.sort_values("VIF", ascending=False)

print(vif_df)
print(f"Features with VIF > 10: {(vif_df['VIF'] > 10).sum()}")
print(f"Features with VIF > 100: {(vif_df['VIF'] > 100).sum()}")

                    Feature           VIF
0               radius_mean  63306.172036
2            perimeter_mean  58123.586079
20             radius_worst   9674.742602
22          perimeter_worst   4487.781270
3                 area_mean   1287.262339
23               area_worst   1138.759252
9    fractal_dimension_mean    629.679874
29  fractal_dimension_worst    423.396723
4           smoothness_mean    393.398166
24         smoothness_worst    375.597155
21            texture_worst    343.004387
1              texture_mean    251.047108
10                radius_se    236.665738
28           symmetry_worst    218.919805
12             perimeter_se    211.396334
5          compactness_mean    200.980354
8             symmetry_mean    184.426558
6            concavity_mean    157.855046
7       concave points_mean    154.241268
27     concave points_worst    148.673180
25        compactness_worst    132.884276
26          concavity_worst     86.310362
13                  area_se     72

In [12]:
# Correlation matrix
corr_matrix = X.corr().abs()

# Upper triangle only
upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)
high_corr_pairs = (
    upper.stack()
         .reset_index()
         .rename(columns={
             "level_0": "Feature_1",
             "level_1": "Feature_2",
             0: "Correlation"
         })
)

high_corr_pairs = high_corr_pairs[
    high_corr_pairs["Correlation"] > 0.95
].sort_values("Correlation", ascending=False)

print(high_corr_pairs.to_string(index=False))
print(f"\nNumber of pairs with |r| > 0.95: {len(high_corr_pairs)}")

      Feature_1       Feature_2  Correlation
    radius_mean  perimeter_mean     0.997855
   radius_worst perimeter_worst     0.993708
    radius_mean       area_mean     0.987357
 perimeter_mean       area_mean     0.986507
   radius_worst      area_worst     0.984015
perimeter_worst      area_worst     0.977578
      radius_se    perimeter_se     0.972794
 perimeter_mean perimeter_worst     0.970387
    radius_mean    radius_worst     0.969539
 perimeter_mean    radius_worst     0.969476
    radius_mean perimeter_worst     0.965137
      area_mean    radius_worst     0.962746
      area_mean      area_worst     0.959213
      area_mean perimeter_worst     0.959120
      radius_se         area_se     0.951830

Number of pairs with |r| > 0.95: 15
